# Task
Scrape global power, military, economic, and geographic data from "https://www.globalfirepower.com/countries-listing.php" and over 50+ related metric pages to create a comprehensive dataset. For each country, extract the Power Index score and specific values for natural gas, coal, land area, border coverage, and other military/economic metrics. Clean the data by removing units and commas, merge all metrics into a single pandas DataFrame, and export the final results to a CSV file named "global_military_economic_geographic_data.csv".

## Scrape Base Country List

### Subtask:
Scrape the main ranking page to extract the master list of countries and their corresponding Power Index scores.


**Concept**:
I will import the necessary libraries and scrape the main ranking page from Global Firepower to create the base DataFrame containing country names and their Power Index scores.

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import pandas as pd


# Constructing the master dictionary for metric URLs
gfp_url_map = {
    'main_ranking': 'https://www.globalfirepower.com/countries-listing.php',
    'land_area': 'https://www.globalfirepower.com/countries-listing-land-area.php',
    'border_coverage': 'https://www.globalfirepower.com/countries-listing-border-coverage.php',
    'coastline_coverage': 'https://www.globalfirepower.com/countries-listing-coastline-coverage.php',
    'waterway_coverage': 'https://www.globalfirepower.com/countries-listing-waterway-coverage.php',
    'defense_spending': 'https://www.globalfirepower.com/countries-listing-defense-spending.php',
    'external_debt': 'https://www.globalfirepower.com/countries-listing-external-debt.php',
    'foreign_exchange_gold': 'https://www.globalfirepower.com/countries-listing-foreign-exchange-reserves.php',
    'purchasing_power_parity': 'https://www.globalfirepower.com/countries-listing-purchasing-power-parity.php',
    'oil_production': 'https://www.globalfirepower.com/countries-listing-oil-production.php',
    'oil_consumption': 'https://www.globalfirepower.com/countries-listing-oil-consumption.php',
    'oil_proven_reserves': 'https://www.globalfirepower.com/countries-listing-oil-proven-reserves.php',
    'natural_gas_production': 'https://www.globalfirepower.com/countries-listing-natural-gas-production.php',
    'natural_gas_consumption': 'https://www.globalfirepower.com/countries-listing-natural-gas-consumption.php',
    'natural_gas_proven_reserves': 'https://www.globalfirepower.com/countries-listing-natural-gas-proven-reserves.php',
    'coal_production': 'https://www.globalfirepower.com/countries-listing-coal-production.php',
    'coal_consumption': 'https://www.globalfirepower.com/countries-listing-coal-consumption.php',
    'coal_proven_reserves': 'https://www.globalfirepower.com/countries-listing-coal-proven-reserves.php',
    'total_population': 'https://www.globalfirepower.com/countries-listing-population-total.php',
    'available_manpower': 'https://www.globalfirepower.com/countries-listing-manpower-available.php',
    'fit_for_service': 'https://www.globalfirepower.com/countries-listing-manpower-fit-for-service.php',
    'reaching_military_age_annually': 'https://www.globalfirepower.com/countries-listing-manpower-reaching-military-age-annually.php',
    'total_military_personnel': 'https://www.globalfirepower.com/countries-listing-military-personnel-total.php',
    'active_personnel': 'https://www.globalfirepower.com/countries-listing-military-personnel-active.php',
    'reserve_personnel': 'https://www.globalfirepower.com/countries-listing-military-personnel-reserve.php',
    'paramilitary_forces': 'https://www.globalfirepower.com/countries-listing-military-personnel-paramilitary.php',
    'air_force_strength': 'https://www.globalfirepower.com/countries-listing-aircraft-total.php',
    'fighters_interceptors': 'https://www.globalfirepower.com/countries-listing-aircraft-fighters.php',
    'attack_aircraft': 'https://www.globalfirepower.com/countries-listing-aircraft-attack.php',
    'transports': 'https://www.globalfirepower.com/countries-listing-aircraft-transports.php',
    'trainers': 'https://www.globalfirepower.com/countries-listing-aircraft-trainers.php',
    'special_mission': 'https://www.globalfirepower.com/countries-listing-aircraft-special-mission.php',
    'tanker_fleet': 'https://www.globalfirepower.com/countries-listing-aircraft-tankers.php',
    'helicopters': 'https://www.globalfirepower.com/countries-listing-aircraft-helicopters.php',
    'attack_helicopters': 'https://www.globalfirepower.com/countries-listing-aircraft-attack-helicopters.php',
    'tanks': 'https://www.globalfirepower.com/countries-listing-tanks.php',
    'armored_vehicles': 'https://www.globalfirepower.com/countries-listing-armored-vehicles.php'
}

print(f"Master URL Dictionary created with {len(gfp_url_map)} entries.")

Master URL Dictionary created with 37 entries.


In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_main_ranking_v8(url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    countries_data = []
    records = soup.find_all('div', class_='recordsetContainer')

    for record in records:
        name_container = record.find('div', class_='longFormName')
        name = name_container.get_text(strip=True) if name_container else None
        
        pi_container = record.find('div', class_='pwrIndxContainer')
        pwr_index = None
        if pi_container:
            # The HTML shows 'PwrIndx: 0.0741'
            pi_text = pi_container.get_text(separator=' ', strip=True).upper()
            if 'PWRINDX' in pi_text:
                # Split by colon and extract the float
                try:
                    val_part = pi_text.split(':')[-1].strip()
                    pwr_index = float(val_part)
                except (ValueError, IndexError):
                    pass
        
        if name:
            countries_data.append({'Country': name, 'Power_Index': pwr_index})
            
    return pd.DataFrame(countries_data)

main_url = gfp_url_map['main_ranking']
df_main = scrape_main_ranking_v8(main_url)

print(f'Successfully scraped {len(df_main)} countries.')
print(df_main.head(5))

Successfully scraped 145 countries.
         Country  Power_Index
0  United States       0.0741
1         Russia       0.0791
2          China       0.0919
3          India       0.1346
4    South Korea       0.1642


In [ ]:
import re
import pandas as pd

def robust_clean_metric(value):
    if pd.isna(value) or value is None:
        return None
    if isinstance(value, (int, float)):
        return value
    
    # Convert to string and remove common non-numeric characters and units
    # Handles $, commas, sq km, mt, bbl, ft, mi, km and whitespace/control chars
    text = str(value)
    cleaned = re.sub(r'[\$\t\r\n,]', '', text)
    cleaned = re.sub(r'(?i)(sq\s*km|mt|bbl|ft|mi|km)', '', cleaned).strip()
    
    # Extract the first valid numeric part (integer or decimal)
    match = re.search(r'[-+]?\d*\.?\d+', cleaned)
    return match.group(0) if match else None

# Identify columns to clean (all except Country and Power_Index)
columns_to_process = [col for col in df_aggregated.columns if col not in ['Country', 'Power_Index']]

# Apply cleaning and convert to numeric
for col in columns_to_process:
    df_aggregated[col] = df_aggregated[col].apply(robust_clean_metric)
    df_aggregated[col] = pd.to_numeric(df_aggregated[col], errors='coerce')

# Verification
print('Data Types and Missing Values Summary:')
print(df_aggregated.info())
print('\nFirst 5 rows of cleaned DataFrame:')
print(df_aggregated.head())

In [16]:
display(df_aggregated.head(5))

,Country,Power_Index,defense_spending_budget,purchasing_power_parity,coastline_coverage,total_aircraft_strength,tank_strength,total_naval_assets
0,United States,0.0741,831500000000,25676000000000,19924,13032,4666,465
1,Russia,0.0791,212638272000,6089000000000,37653,4237,5630,747
2,China,0.0919,303000000000,33598000000000,14500,3529,5870,841
3,India,0.1346,109000000000,14244000000000,7000,2183,3913,343
4,South Korea,0.1642,44800000000,2607000000000,2413,1540,1831,215


## Update Extended URL Map

### Subtask:
Expand the `gfp_url_map` with corrected and additional URLs for comprehensive scraping.


**Reasoning**:
I will update the gfp_url_map with the correct URL patterns for the requested metrics to ensure they can be scraped successfully in the next steps.



In [20]:
# Expanding the URL map with verified/corrected patterns for the requested metrics
gfp_url_map = {
    'main_ranking': 'https://www.globalfirepower.com/countries-listing.php',
    'total_population': 'https://www.globalfirepower.com/population-total.php',
    'available_manpower': 'https://www.globalfirepower.com/manpower-available.php',
    'oil_production': 'https://www.globalfirepower.com/oil-production.php',
    'natural_gas_production': 'https://www.globalfirepower.com/natural-gas-production.php',
    'coal_production': 'https://www.globalfirepower.com/coal-production.php',
    'square_land_area': 'https://www.globalfirepower.com/land-area-total.php',
    'defense_spending': 'https://www.globalfirepower.com/defense-spending-budget.php',
    'purchasing_power_parity': 'https://www.globalfirepower.com/purchasing-power-parity.php',
    'coastline_coverage': 'https://www.globalfirepower.com/coastline-coverage.php',
    'border_coverage': 'https://www.globalfirepower.com/border-coverage.php',
    'total_aircraft_strength': 'https://www.globalfirepower.com/aircraft-total.php',
    'tank_strength': 'https://www.globalfirepower.com/armor-tanks-total.php',
    'total_naval_assets': 'https://www.globalfirepower.com/navy-ships.php'
}

print(f'Updated gfp_url_map with {len(gfp_url_map)} target metric URLs.')

Updated gfp_url_map with 14 target metric URLs.


## Scrape Additional Metrics

### Subtask:
Iteratively scrape the supplemental metric pages from the updated `gfp_url_map` and merge them into the master DataFrame.


**Reasoning**:
I will redefine the scraping function to target the 'valueContainer' class and then iterate through the updated URL map to merge the new metrics into the existing aggregated DataFrame.



In [21]:
import time
import requests
from bs4 import BeautifulSoup
import pandas as pd

def scrape_gfp_metric_v5(metric_name, url):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    try:
        response = requests.get(url, headers=headers, timeout=15)
        if response.status_code != 200:
            return pd.DataFrame(columns=['Country', metric_name])
    except Exception:
        return pd.DataFrame(columns=['Country', metric_name])

    soup = BeautifulSoup(response.content, 'html.parser')
    metric_data = []
    records = soup.find_all('div', class_='recordsetContainer')

    for record in records:
        name_tag = record.find('div', class_='longFormName')
        if not name_tag:
            continue
        name = name_tag.get_text(strip=True)

        # The metric value is located inside a div with class 'valueContainer'
        val_container = record.find('div', class_='valueContainer')
        val = val_container.get_text(strip=True) if val_container else None

        if name:
            metric_data.append({'Country': name, metric_name: val})

    return pd.DataFrame(metric_data)

# Iterate through the updated gfp_url_map
for metric_name, url in gfp_url_map.items():
    if metric_name == 'main_ranking':
        continue
    
    # Check if the column already exists in df_aggregated to avoid duplicate merges
    if metric_name in df_aggregated.columns:
        print(f'Metric {metric_name} already exists in DataFrame. Skipping...')
        continue

    print(f'Scraping {metric_name}...', end=' ')
    df_temp = scrape_gfp_metric_v5(metric_name, url)
    
    if not df_temp.empty:
        # Left merge into the existing aggregated dataframe
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Success ({len(df_temp)} records)')
    else:
        print('Failed or No Data')
    
    # Respect rate limits
    time.sleep(1)

print('\nAggregated Data Preview:')
print(df_aggregated.head())
print(f'Final Shape: {df_aggregated.shape}')

Scraping total_population... Failed or No Data
Scraping available_manpower... Failed or No Data
Scraping oil_production... Failed or No Data
Scraping natural_gas_production... Failed or No Data
Scraping coal_production... Failed or No Data
Scraping square_land_area... Failed or No Data
Scraping defense_spending... Success (145 records)
Scraping purchasing_power_parity... Success (145 records)
Scraping coastline_coverage... Success (145 records)
Scraping border_coverage... Success (145 records)
Scraping total_aircraft_strength... Success (145 records)
Scraping tank_strength... Success (145 records)
Scraping total_naval_assets... Success (145 records)

Aggregated Data Preview:
         Country  Power_Index  \
0  United States       0.0741   
1         Russia       0.0791   
2          China       0.0919   
3          India       0.1346   
4    South Korea       0.1642   

                                    defense_spending  \
0  $                        \t\t\t\t\t\t\t831,500...   
1  $ 

## Clean New Metric Columns

### Subtask:
Apply the robust_clean_metric function to the new columns to remove units and symbols, then convert them to numeric types.


**Reasoning**:
I will apply the robust_clean_metric function to clean the aggregated DataFrame columns, removing non-numeric characters and converting the values to numeric types to ensure data consistency.



In [22]:
import re
import pandas as pd

def robust_clean_metric(value):
    if pd.isna(value) or value is None:
        return None
    if isinstance(value, (int, float)):
        return value
    text = str(value)
    cleaned = re.sub(r'[\$\t\r\n,]', '', text)
    cleaned = re.sub(r'(?i)(sq\s*km|mt|bbl|ft|mi|km)', '', cleaned).strip()
    match = re.search(r'[-+]?\d*\.?\d+', cleaned)
    return match.group(0) if match else None

# Identify columns to clean (all except Country and Power_Index)
columns_to_process = [col for col in df_aggregated.columns if col not in ['Country', 'Power_Index']]

# Apply cleaning and convert to numeric
for col in columns_to_process:
    df_aggregated[col] = df_aggregated[col].apply(robust_clean_metric)
    df_aggregated[col] = pd.to_numeric(df_aggregated[col], errors='coerce')

# Verification
print('Data Types and Missing Values Summary:')
print(df_aggregated.info())
print('\nFirst 5 rows of cleaned DataFrame:')
print(df_aggregated.head())

Data Types and Missing Values Summary:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Country                  145 non-null    object 
 1   Power_Index              145 non-null    float64
 2   defense_spending         145 non-null    int64  
 3   purchasing_power_parity  145 non-null    int64  
 4   coastline_coverage       145 non-null    int64  
 5   border_coverage          145 non-null    int64  
 6   total_aircraft_strength  145 non-null    int64  
 7   tank_strength            145 non-null    int64  
 8   total_naval_assets       145 non-null    int64  
dtypes: float64(1), int64(7), object(1)
memory usage: 10.3+ KB
None

First 5 rows of cleaned DataFrame:
         Country  Power_Index  defense_spending  purchasing_power_parity  \
0  United States       0.0741      831500000000           25676000000000   
1      

**Reasoning**:
The cleaning subtask has been completed, with all metric columns successfully converted to numeric types and verified. I will now overwrite the final CSV file with the current expanded DataFrame as requested in the task description.



In [23]:
display(df_aggregated.head(5))

,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215


In [24]:
import pandas as pd

target_url = 'https://www.globalfirepower.com/aircraft-total-fighters.php'
metric_name = 'fighter_aircraft'

print(f'Scraping {metric_name}...')
df_fighters = scrape_gfp_metric_v5(metric_name, target_url)

if not df_fighters.empty:
    # Clean data
    df_fighters[metric_name] = df_fighters[metric_name].apply(robust_clean_metric)
    df_fighters[metric_name] = pd.to_numeric(df_fighters[metric_name], errors='coerce')

    # Merge
    df_aggregated = pd.merge(df_aggregated, df_fighters, on='Country', how='left')
    
    # Update CSV
    df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
    
    print(f'Successfully added {metric_name}. New shape: {df_aggregated.shape}')
    display(df_aggregated.head())
else:
    print(f'Failed to scrape {metric_name}. Please verify the URL.')

Scraping fighter_aircraft...
Successfully added fighter_aircraft. New shape: (145, 10)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242


In [25]:
import pandas as pd
import time

# Define the metrics to scrape in this batch based on earlier shared links
additional_metrics = {
    'attack_aircraft': 'https://www.globalfirepower.com/aircraft-total-attack-types.php',
    'transport_aircraft': 'https://www.globalfirepower.com/aircraft-total-transports.php'
}

for metric_name, target_url in additional_metrics.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Merge
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')
    
    time.sleep(1)

# Update CSV
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'Final shape after additions: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping attack_aircraft...
Successfully added attack_aircraft.
Scraping transport_aircraft...
Successfully added transport_aircraft.
Final shape after additions: (145, 12)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,attack_aircraft,transport_aircraft
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,926,917
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,698,458
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,371,287
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,124,277
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,98,40


In [26]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Based on the site structure, total military personnel is often 'active-military-manpower.php' or similar.
# Let's try the most common verified URL for total military personnel and the one we just found.
target_urls = [
    'https://www.globalfirepower.com/active-military-manpower.php',
    'https://www.globalfirepower.com/military-personnel-active.php',
    'https://www.globalfirepower.com/available-military-manpower.php'
]

for url in target_urls:
    print(f'Trying URL: {url}')
    df_manpower = scrape_gfp_metric_v5('total_military_manpower', url)
    if not df_manpower.empty:
        print(f'Successfully scraped {len(df_manpower)} records from {url}.')
        break

if df_manpower.empty:
    print('All predefined manpower URLs failed. Checking the main page again for any country-listing style links for personnel.')
    headers = {'User-Agent': 'Mozilla/5.0'}
    resp = requests.get('https://www.globalfirepower.com/countries-listing.php', headers=headers)
    soup = BeautifulSoup(resp.content, 'html.parser')
    # Looking for the specific keyword in any link
    for a in soup.find_all('a', href=True):
        if 'personnel' in a['href'] or 'manpower' in a['href']:
            print(f'Found potential link: {a["href"]}')

print(df_manpower.head())

Trying URL: https://www.globalfirepower.com/active-military-manpower.php
Successfully scraped 145 records from https://www.globalfirepower.com/active-military-manpower.php.
         Country total_military_manpower
0          China               2,035,000
1          India               1,431,000
2  United States               1,333,030
3         Russia               1,320,000
4    North Korea               1,320,000


**Reasoning**:
The manpower data has been successfully extracted from the verified URL. I will now clean the raw numeric strings using the existing robust_clean_metric function and merge this new data into the main aggregated DataFrame.



In [29]:
# Cleaning the newly scraped manpower data
df_manpower['total_military_manpower'] = df_manpower['total_military_manpower'].apply(robust_clean_metric)
df_manpower['total_military_manpower'] = pd.to_numeric(df_manpower['total_military_manpower'], errors='coerce')

# Merging the new metric into the aggregated DataFrame
df_aggregated = pd.merge(df_aggregated, df_manpower, on='Country', how='left')

# Overwriting the CSV file with the updated data
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)

print('Successfully merged total_military_manpower and updated CSV.')
print(f'New DataFrame shape: {df_aggregated.shape}')


Successfully merged total_military_manpower and updated CSV.
New DataFrame shape: (145, 14)


In [28]:
display(df_aggregated.head(5))

,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,attack_aircraft,transport_aircraft,total_military_manpower
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,926,917,1333030
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,698,458,1320000
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,371,287,2035000
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,124,277,1431000
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,98,40,450000


In [30]:
import pandas as pd
import time

# Metrics provided by the user
aircraft_batch = {
    'trainer_aircraft': 'https://www.globalfirepower.com/aircraft-total-trainers.php',
    'special_mission_aircraft': 'https://www.globalfirepower.com/aircraft-total-special-mission.php',
    'tanker_aircraft': 'https://www.globalfirepower.com/aircraft-total-tanker-fleet.php'
}

for metric_name, target_url in aircraft_batch.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Merge
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')
    
    time.sleep(1)

# Update CSV
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'New DataFrame shape: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping trainer_aircraft...
Successfully added trainer_aircraft.
Scraping special_mission_aircraft...
Successfully added special_mission_aircraft.
Scraping tanker_aircraft...
Successfully added tanker_aircraft.
New DataFrame shape: (145, 17)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,attack_aircraft,transport_aircraft,total_military_manpower_x,total_military_manpower_y,trainer_aircraft,special_mission_aircraft,tanker_aircraft
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,926,917,1333030,1333030,2610,611,610
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,698,458,1320000,1320000,530,137,18
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,371,287,2035000,2035000,401,105,9
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,124,277,1431000,1431000,334,72,6
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,98,40,450000,450000,335,35,4


In [35]:
import pandas as pd
import time

# Define the metrics to scrape in this batch based on provided links
helos_tanks_batch = {
    'total_military_helicopters': 'https://www.globalfirepower.com/aircraft-helicopters-total.php',
    'attack_helicopters': 'https://www.globalfirepower.com/aircraft-helicopters-attack.php',
    'tanks': 'https://www.globalfirepower.com/armor-tanks-total.php'
}

for metric_name, target_url in helos_tanks_batch.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data using existing function
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Merge into the main DataFrame
        # If the metric name already exists (like 'tanks' might from previous runs), drop it first or update
        if metric_name in df_aggregated.columns:
            df_aggregated = df_aggregated.drop(columns=[metric_name])
            
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')

    time.sleep(1)

# Update the CSV file
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'New DataFrame shape: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping total_military_helicopters...
Successfully added total_military_helicopters.
Scraping attack_helicopters...
Successfully added attack_helicopters.
Scraping tanks...
Successfully added tanks.
New DataFrame shape: (145, 20)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,attack_aircraft,transport_aircraft,total_military_manpower_x,total_military_manpower_y,trainer_aircraft,special_mission_aircraft,tanker_aircraft,total_military_helicopters,attack_helicopters,tanks
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,926,917,1333030,1333030,2610,611,610,5913,1024,4666
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,698,458,1320000,1320000,530,137,18,1643,556,5630
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,371,287,2035000,2035000,401,105,9,1007,281,5870
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,124,277,1431000,1431000,334,72,6,594,79,3913
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,98,40,450000,450000,335,35,4,827,113,1831


In [36]:
import pandas as pd
import time

# Define the metrics to scrape in this batch
land_equipment_batch = {
    'armored_fighting_vehicles': 'https://www.globalfirepower.com/armor-apc-total.php',
    'self_propelled_artillery': 'https://www.globalfirepower.com/armor-self-propelled-guns-total.php',
    'towed_artillery': 'https://www.globalfirepower.com/armor-towed-artillery-total.php',
    'rocket_projectors': 'https://www.globalfirepower.com/armor-mlrs-total.php'
}

for metric_name, target_url in land_equipment_batch.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data using existing function
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Drop metric if it already exists to avoid duplication
        if metric_name in df_aggregated.columns:
            df_aggregated = df_aggregated.drop(columns=[metric_name])

        # Merge into the main DataFrame
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')

    time.sleep(1)

# Update the CSV file
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'New DataFrame shape: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping armored_fighting_vehicles...
Successfully added armored_fighting_vehicles.
Scraping self_propelled_artillery...
Successfully added self_propelled_artillery.
Scraping towed_artillery...
Successfully added towed_artillery.
Scraping rocket_projectors...
Successfully added rocket_projectors.
New DataFrame shape: (145, 24)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,...,trainer_aircraft,special_mission_aircraft,tanker_aircraft,total_military_helicopters,attack_helicopters,tanks,armored_fighting_vehicles,self_propelled_artillery,towed_artillery,rocket_projectors
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,...,2610,611,610,5913,1024,4666,409660,1521,1878,1731
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,...,530,137,18,1643,556,5630,126512,3603,5920,2486
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,...,401,105,9,1007,281,5870,152040,2940,1400,2770
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,...,334,72,6,594,79,3913,163554,100,5640,300
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,...,335,35,4,827,113,1831,117460,2780,5800,525


In [37]:
import pandas as pd
import time

# Define the naval metrics to scrape in this batch
navy_batch_1 = {
    'total_naval_fleet': 'https://www.globalfirepower.com/navy-ships.php',
    'total_naval_fleet_tonnage_mt': 'https://www.globalfirepower.com/navy-force-by-tonnage.php',
    'aircraft_carriers': 'https://www.globalfirepower.com/navy-aircraft-carriers.php',
    'helicopter_carriers': 'https://www.globalfirepower.com/navy-helo-carriers.php',
    'submarines': 'https://www.globalfirepower.com/navy-submarines.php',
    'destroyers': 'https://www.globalfirepower.com/navy-destroyers.php'
}

for metric_name, target_url in navy_batch_1.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data using existing function
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Drop metric if it already exists to avoid duplication
        if metric_name in df_aggregated.columns:
            df_aggregated = df_aggregated.drop(columns=[metric_name])

        # Merge into the main DataFrame
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')

    time.sleep(1)

# Update the CSV file
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'New DataFrame shape: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping total_naval_fleet...
Successfully added total_naval_fleet.
Scraping total_naval_fleet_tonnage_mt...
Successfully added total_naval_fleet_tonnage_mt.
Scraping aircraft_carriers...
Successfully added aircraft_carriers.
Scraping helicopter_carriers...
Successfully added helicopter_carriers.
Scraping submarines...
Successfully added submarines.
Scraping destroyers...
Successfully added destroyers.
New DataFrame shape: (145, 30)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,...,armored_fighting_vehicles,self_propelled_artillery,towed_artillery,rocket_projectors,total_naval_fleet,total_naval_fleet_tonnage_mt,aircraft_carriers,helicopter_carriers,submarines,destroyers
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,...,409660,1521,1878,1731,465,8265799.0,11,9,66,83
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,...,126512,3603,5920,2486,747,1426539.0,1,0,66,13
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,...,152040,2940,1400,2770,841,3192411.0,3,4,61,53
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,...,163554,100,5640,300,343,631989.0,2,0,18,13
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,...,117460,2780,5800,525,215,427946.0,0,2,22,14


In [38]:
import pandas as pd
import time

# Define the metrics to scrape in this batch
navy_econ_batch = {
    'frigates': 'https://www.globalfirepower.com/navy-frigates.php',
    'corvettes': 'https://www.globalfirepower.com/navy-corvettes.php',
    'coastal_patrol_craft': 'https://www.globalfirepower.com/navy-patrol-coastal-craft.php',
    'defense_budget_usd': 'https://www.globalfirepower.com/defense-spending-budget.php',
    'external_debt_usd': 'https://www.globalfirepower.com/external-debt-by-country.php'
}

for metric_name, target_url in navy_econ_batch.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data using existing function
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Drop metric if it already exists to avoid duplication
        if metric_name in df_aggregated.columns:
            df_aggregated = df_aggregated.drop(columns=[metric_name])

        # Merge into the main DataFrame
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')

    time.sleep(1)

# Update the CSV file
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'New DataFrame shape: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping frigates...
Successfully added frigates.
Scraping corvettes...
Successfully added corvettes.
Scraping coastal_patrol_craft...
Successfully added coastal_patrol_craft.
Scraping defense_budget_usd...
Successfully added defense_budget_usd.
Scraping external_debt_usd...
Successfully added external_debt_usd.
New DataFrame shape: (145, 35)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,...,total_naval_fleet_tonnage_mt,aircraft_carriers,helicopter_carriers,submarines,destroyers,frigates,corvettes,coastal_patrol_craft,defense_budget_usd,external_debt_usd
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,...,8265799.0,11,9,66,83,0,27,0,831500000000,24538900710000
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,...,1426539.0,1,0,66,13,12,79,70,212638272000,226475750000
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,...,3192411.0,3,4,61,53,46,50,150,303000000000,488114000000
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,...,631989.0,2,0,18,13,18,21,146,109000000000,212728000000
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,...,427946.0,0,2,22,14,18,3,96,44800000000,553871450000


In [39]:
import pandas as pd
import time

# Define the metrics to scrape in this batch
infra_reserves_batch = {
    'foreign_exchange_gold_reserves': 'https://www.globalfirepower.com/reserves-of-foreign-exchange-and-gold.php',
    'total_serviceable_airports': 'https://www.globalfirepower.com/major-serviceable-airports-by-country.php',
    'major_ports_and_terminals': 'https://www.globalfirepower.com/major-ports-and-terminals.php',
    'total_merchant_marine_fleet': 'https://www.globalfirepower.com/merchant-marine-strength-by-country.php'
}

for metric_name, target_url in infra_reserves_batch.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Drop metric if it already exists to avoid duplication
        if metric_name in df_aggregated.columns:
            df_aggregated = df_aggregated.drop(columns=[metric_name])

        # Merge
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')

    time.sleep(1)

# Update CSV
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'New DataFrame shape: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping foreign_exchange_gold_reserves...
Successfully added foreign_exchange_gold_reserves.
Scraping total_serviceable_airports...
Successfully added total_serviceable_airports.
Scraping major_ports_and_terminals...
Successfully added major_ports_and_terminals.
Scraping total_merchant_marine_fleet...
Successfully added total_merchant_marine_fleet.
New DataFrame shape: (145, 39)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,...,destroyers,frigates,corvettes,coastal_patrol_craft,defense_budget_usd,external_debt_usd,foreign_exchange_gold_reserves,total_serviceable_airports,major_ports_and_terminals,total_merchant_marine_fleet
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,...,83,0,27,0,831500000000,24538900710000,910037000000,16116,666,3533
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,...,13,12,79,70,212638272000,226475750000,597217000000,905,67,2910
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,...,53,46,50,150,303000000000,488114000000,3456000000000,552,66,8314
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,...,13,18,21,146,109000000000,212728000000,643043000000,315,56,1859
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,...,14,18,3,96,44800000000,553871450000,418219000000,92,15,2149


In [40]:
import pandas as pd
import time

# Define the metrics to scrape in this batch
logistics_oil_batch = {
    'railway_coverage_km': 'https://www.globalfirepower.com/railway-coverage.php',
    'roadway_coverage_km': 'https://www.globalfirepower.com/roadway-coverage.php',
    'oil_production_bbl': 'https://www.globalfirepower.com/oil-production-by-country.php',
    'oil_consumption_bbl': 'https://www.globalfirepower.com/oil-consumption-by-country.php',
    'proven_oil_reserves_bbl': 'https://www.globalfirepower.com/proven-oil-reserves-by-country.php'
}

for metric_name, target_url in logistics_oil_batch.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data using existing function
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Drop metric if it already exists to avoid duplication
        if metric_name in df_aggregated.columns:
            df_aggregated = df_aggregated.drop(columns=[metric_name])

        # Merge into the main DataFrame
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')

    time.sleep(1)

# Update the CSV file
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'New DataFrame shape: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping railway_coverage_km...
Successfully added railway_coverage_km.
Scraping roadway_coverage_km...
Successfully added roadway_coverage_km.
Scraping oil_production_bbl...
Successfully added oil_production_bbl.
Scraping oil_consumption_bbl...
Successfully added oil_consumption_bbl.
Scraping proven_oil_reserves_bbl...
Successfully added proven_oil_reserves_bbl.
New DataFrame shape: (145, 44)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,...,external_debt_usd,foreign_exchange_gold_reserves,total_serviceable_airports,major_ports_and_terminals,total_merchant_marine_fleet,railway_coverage_km,roadway_coverage_km,oil_production_bbl,oil_consumption_bbl,proven_oil_reserves_bbl
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,...,24538900710000,910037000000,16116,666,3533,293564,6586610,20953000,20307000,38212000000
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,...,226475750000,597217000000,905,67,2910,85494,1283387,10879000,3863000,80000000000
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,...,488114000000,3456000000000,552,66,8314,150000,5200000,4984000,16189000,26023000000
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,...,212728000000,643043000000,315,56,1859,65554,6371847,822000,5271000,4605000000
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,...,553871450000,418219000000,92,15,2149,3979,100428,38000,2542000,0


In [41]:
import pandas as pd
import time

# Define the metrics to scrape in this final batch
energy_geo_batch = {
    'natural_gas_production_cum': 'https://www.globalfirepower.com/natural-gas-production-by-country.php',
    'natural_gas_consumption_cum': 'https://www.globalfirepower.com/natural-gas-consumption-by-country.php',
    'proven_natural_gas_reserves_cum': 'https://www.globalfirepower.com/proven-natural-gas-reserves-by-country.php',
    'coal_production_cum': 'https://www.globalfirepower.com/coal-production-by-country.php',
    'coal_consumption_mt': 'https://www.globalfirepower.com/coal-consumption-by-country.php',
    'proven_coal_reserves_cum': 'https://www.globalfirepower.com/proven-coal-reserves-by-country.php',
    'total_land_area_sq_km': 'https://www.globalfirepower.com/square-land-area.php'
}

for metric_name, target_url in energy_geo_batch.items():
    print(f'Scraping {metric_name}...')
    df_temp = scrape_gfp_metric_v5(metric_name, target_url)

    if not df_temp.empty:
        # Clean data using existing function
        df_temp[metric_name] = df_temp[metric_name].apply(robust_clean_metric)
        df_temp[metric_name] = pd.to_numeric(df_temp[metric_name], errors='coerce')

        # Drop metric if it already exists to avoid duplication
        if metric_name in df_aggregated.columns:
            df_aggregated = df_aggregated.drop(columns=[metric_name])

        # Merge into the main DataFrame
        df_aggregated = pd.merge(df_aggregated, df_temp, on='Country', how='left')
        print(f'Successfully added {metric_name}.')
    else:
        print(f'Failed to scrape {metric_name}.')

    time.sleep(1)

# Final Update to the CSV file
df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)
print(f'Final DataFrame shape: {df_aggregated.shape}')
display(df_aggregated.head())

Scraping natural_gas_production_cum...
Successfully added natural_gas_production_cum.
Scraping natural_gas_consumption_cum...
Successfully added natural_gas_consumption_cum.
Scraping proven_natural_gas_reserves_cum...
Successfully added proven_natural_gas_reserves_cum.
Scraping coal_production_cum...
Successfully added coal_production_cum.
Scraping coal_consumption_mt...
Successfully added coal_consumption_mt.
Scraping proven_coal_reserves_cum...
Successfully added proven_coal_reserves_cum.
Scraping total_land_area_sq_km...
Successfully added total_land_area_sq_km.
Final DataFrame shape: (145, 51)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,...,oil_production_bbl,oil_consumption_bbl,proven_oil_reserves_bbl,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,...,20953000,20307000,38212000000,1029000000000,914301000000,13402000000000,534234000,495156000,247883000000,9833517
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,...,10879000,3863000,80000000000,617830000000,472239000000,47805000000000,531130000,290763000,162166000000,17098242
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,...,4984000,16189000,26023000000,225341000000,366160000000,6654000000000,4805000000,5191000000,157041000000,9596960
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,...,822000,5271000,4605000000,33170000000,58867000000,1381000000000,1020000000,1262000000,127727000000,3287263
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,...,38000,2542000,0,55127000,59480000000,7079000000,16081000,136817000,326000000,99720


In [42]:
import pandas as pd

target_url = 'https://www.globalfirepower.com/waterway-coverage.php'
metric_name = 'waterway_coverage_km'

print(f'Scraping {metric_name}...')
df_waterway = scrape_gfp_metric_v5(metric_name, target_url)

if not df_waterway.empty:
    # Clean data
    df_waterway[metric_name] = df_waterway[metric_name].apply(robust_clean_metric)
    df_waterway[metric_name] = pd.to_numeric(df_waterway[metric_name], errors='coerce')

    # Merge
    if metric_name in df_aggregated.columns:
        df_aggregated = df_aggregated.drop(columns=[metric_name])
    df_aggregated = pd.merge(df_aggregated, df_waterway, on='Country', how='left')

    # Update CSV
    df_aggregated.to_csv('global_military_economic_geographic_data.csv', index=False)

    print(f'Successfully added {metric_name}. New shape: {df_aggregated.shape}')
    display(df_aggregated.head())
else:
    print(f'Failed to scrape {metric_name}. Please verify the URL.')

Scraping waterway_coverage_km...
Successfully added waterway_coverage_km. New shape: (145, 52)


,Country,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,...,oil_consumption_bbl,proven_oil_reserves_bbl,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,waterway_coverage_km
0,United States,0.0741,831500000000,25676000000000,19924,12002,13032,4666,465,1791,...,20307000,38212000000,1029000000000,914301000000,13402000000000,534234000,495156000,247883000000,9833517,41009
1,Russia,0.0791,212638272000,6089000000000,37653,22407,4237,5630,747,861,...,3863000,80000000000,617830000000,472239000000,47805000000000,531130000,290763000,162166000000,17098242,102000
2,China,0.0919,303000000000,33598000000000,14500,22457,3529,5870,841,1443,...,16189000,26023000000,225341000000,366160000000,6654000000000,4805000000,5191000000,157041000000,9596960,27700
3,India,0.1346,109000000000,14244000000000,7000,13888,2183,3913,343,476,...,5271000,4605000000,33170000000,58867000000,1381000000000,1020000000,1262000000,127727000000,3287263,14500
4,South Korea,0.1642,44800000000,2607000000000,2413,237,1540,1831,215,242,...,2542000,0,55127000,59480000000,7079000000,16081000,136817000,326000000,99720,1600


In [43]:
print("Dataset Ready for Cleaning:")
print(df_aggregated.info())
display(df_aggregated.describe())

Dataset Ready for Cleaning:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145 entries, 0 to 144
Data columns (total 52 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Country                          145 non-null    object 
 1   Power_Index                      145 non-null    float64
 2   defense_spending                 145 non-null    int64  
 3   purchasing_power_parity          145 non-null    int64  
 4   coastline_coverage               145 non-null    int64  
 5   border_coverage                  145 non-null    int64  
 6   total_aircraft_strength          145 non-null    int64  
 7   tank_strength                    145 non-null    int64  
 8   total_naval_assets               145 non-null    int64  
 9   fighter_aircraft                 145 non-null    int64  
 10  attack_aircraft                  145 non-null    int64  
 11  transport_aircraft               145 non-null    int64  

,Power_Index,defense_spending,purchasing_power_parity,coastline_coverage,border_coverage,total_aircraft_strength,tank_strength,total_naval_assets,fighter_aircraft,attack_aircraft,...,oil_consumption_bbl,proven_oil_reserves_bbl,natural_gas_production_cum,natural_gas_consumption_cum,proven_natural_gas_reserves_cum,coal_production_cum,coal_consumption_mt,proven_coal_reserves_cum,total_land_area_sq_km,waterway_coverage_km
count,145.000000,1.450000e+02,1.450000e+02,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,145.000000,...,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,1.450000e+02,145.000000
mean,1.642772,1.973190e+10,1.176933e+12,4695.979310,3583.358621,359.041379,501.675862,73.882759,70.951724,26.489655,...,6.947103e+05,1.151113e+10,2.792901e+10,2.768592e+10,1.377471e+12,6.465795e+07,6.465180e+07,8.000759e+09,9.144517e+05,4345.241379
std,1.135182,7.671986e+10,3.765151e+12,18166.860635,3717.060550,1191.810935,1040.944569,121.408087,211.892017,101.512801,...,2.254375e+06,4.308222e+10,1.060376e+11,9.339564e+10,5.483828e+12,4.168413e+08,4.450332e+08,3.209863e+10,2.178886e+06,11728.517195
min,0.074100,1.820000e+07,2.534000e+09,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,2.000000e+03,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,7.190000e+02,0.000000
25%,0.651700,6.800000e+08,6.541500e+10,47.000000,1253.000000,25.000000,0.000000,5.000000,0.000000,0.000000,...,3.700000e+04,0.000000e+00,0.000000e+00,2.986000e+06,0.000000e+00,0.000000e+00,1.400000e+04,0.000000e+00,8.360000e+04,0.000000
50%,1.543200,2.485350e+09,2.187620e+11,713.000000,2524.000000,99.000000,132.000000,26.000000,11.000000,0.000000,...,1.150000e+05,4.099300e+07,8.020000e+07,2.635000e+09,8.495000e+09,2.700000e+04,1.096000e+06,4.000000e+07,2.742000e+05,825.000000
75%,2.396900,9.174000e+09,7.459940e+11,2640.000000,5002.000000,271.000000,355.000000,100.000000,47.000000,17.000000,...,4.300000e+05,1.500000e+09,1.227000e+10,2.071900e+10,2.406930e+11,4.476000e+06,9.720000e+06,1.410000e+09,7.561020e+05,2800.000000
max,5.799100,8.315000e+11,3.359800e+13,202080.000000,22457.000000,13032.000000,5870.000000,841.000000,1791.000000,926.000000,...,2.030700e+07,3.038060e+11,1.029000e+12,9.143010e+11,4.780500e+13,4.805000e+09,5.191000e+09,2.478830e+11,1.709824e+07,102000.000000


In [44]:
import pandas as pd

# Save to CSV file
df_aggregated.to_csv("scraped_military_data.csv", index=False)

print("File saved successfully!")

File saved successfully!
